# Day 2 — Linear Regression

## Step 1 -Train a LinearRegression Model

In [64]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [65]:
house_prices = pd.read_csv("American_Housing_Data.csv")
house_prices.shape

(39981, 14)

In [66]:
house_prices.isnull().sum()
house_prices.dropna(inplace=True)
house_prices.shape

(39979, 14)

2 Rows with missing values were dropped to ensure accuracy.

In [67]:
x = house_prices.drop(["Price","Address","Zip Code","City","County"], axis = 1)
x = pd.get_dummies(x, drop_first=True)
y = house_prices["Price"]

    Zip Code and Address are categorical variables that cannot be used in linear regression since the zip code represents a categorical code and address represents a unique location for every single house.
    Cities were dropped as they create ann overhead, due to dummy columns, making it unreliable for the model. 

In [68]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [69]:
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

    An empty/untrained model was created and the training (featers & target) data  was used to fit a prediciton line.
    Testing features were used to generate predictions.

## Step 2 - Model coefficients

In [70]:
print(model.coef_)
print(model.intercept_)

[-6.46337452e+04  1.35277887e+05  3.11298474e+02 -2.26217767e+00
  3.72181225e+01  4.04497309e+00 -1.43085458e+04 -4.06964721e+03
  3.52894428e+05 -2.88705226e+04  1.78502283e+05  2.86310298e+05
 -9.48179047e+03 -1.24958026e+05 -5.19664484e+04 -2.31611599e+05
 -9.43094462e+04 -1.56589218e+05 -7.65448214e+04  1.66011255e+05
 -3.50069166e+04 -1.59199036e+05 -2.32990520e+05 -1.00332496e+05
 -1.04026693e+05  2.28148547e+05 -3.82423622e+04  2.74826760e+04
 -1.99337758e+05 -5.81012121e+04 -5.10620787e+03  3.84759363e+04
 -1.61896038e+05  4.58180902e+04  1.81948992e+05  1.11623619e+05]
-448876.12460609665


In [71]:
coeff = pd.Series(model.coef_, index = X_train.columns)
print(coeff)
print("Intercept:", model.intercept_)

Beds                          -64633.745193
Baths                         135277.886959
Living Space                     311.298474
Zip Code Population               -2.262178
Zip Code Density                  37.218123
Median Household Income            4.044973
Latitude                      -14308.545840
Longitude                      -4069.647213
State_California              352894.428453
State_Colorado                -28870.522557
State_District of Columbia    178502.283251
State_Florida                 286310.297549
State_Georgia                  -9481.790474
State_Illinois               -124958.026490
State_Indiana                 -51966.448367
State_Kansas                 -231611.598666
State_Kentucky                -94309.446212
State_Louisiana              -156589.218361
State_Maryland                -76544.821407
State_Michigan                166011.254688
State_Minnesota               -35006.916626
State_Missouri               -159199.035783
State_Nebraska               -23

Because those numbers are in different scales (e.g. # beds ranges between 1-6, living space is measured on thousands) they were rescaled using the following code in order to be able to fairly compare them .

In [72]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_Scaled = scaler.transform(X_test)

model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)
scaled_coeffs = pd.Series(model_scaled.coef_, index=X_train.columns)
#pd.set_option('display.max_rows', None)
print(scaled_coeffs.sort_values(key=abs, ascending=False))

Living Space                  379233.831789
Latitude                     -337167.512888
State_California              218125.584218
Median Household Income       191451.598529
State_Washington              173798.507029
Baths                         168257.391307
State_Oregon                  119057.725230
State_Michigan                119012.701407
State_Minnesota               116136.557115
State_Wisconsin               109244.760191
State_New York                100290.652730
Zip Code Density               99991.504092
State_District of Columbia     94032.408066
State_Colorado                 91512.576073
Beds                          -82980.732672
State_Ohio                     80713.275183
Longitude                     -78117.596443
State_Pennsylvania             74044.039772
State_Indiana                  68398.143374
State_Illinois                 66703.507176
State_Texas                   -66442.316098
State_Tennessee                57946.726645
State_Maryland                 5

Observations:

    The feature with largest absolute (standardized) coefficient value is the living space - the larger the house area the higher the price.
    Latitude being second explains the geographical effect on house prices, with hifher latitude being associated with lower prices.
    Increaseing in the number of Baths increases the house value while more bedrooms lowers the predicted price.


## Step 3 - Model Evaluation

In [73]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [74]:
print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions))) # RMSE
print("R²:", r2_score(y_test, predictions))

MAE: 292289.4320353923
RMSE: 793342.8870673801
R²: 0.4255048195994674


    1- MAE : on average, the model's predictions are off by 292K , a quite large error estimate since the prices mostly fall within the 100 - 900 thousands range.
    2- RMSE : There is a big difference in comparison to the RMSE which is an indicator of inaccurate predictions. RMSE penalizes large errors more heabily yhan small ones, this gap shows that the model is making large mispredictions even though its typical error is smaller.
    3- R² : Since the value is closer to 0 than 1, the model does not majorly describe house price differences - only 43% of the variance is  in prices. The remaining 57% is affected by uncaptured features like city, state, etc.

## Step 4

In [76]:
from sklearn.metrics import mean_squared_error

baselined_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=np.float64)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baselined_pred))
model_rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("Baseline RMSE (prediciting mean):", baseline_rmse)
print("Model RMSE:", model_rmse)
print("Improvement:", baseline_rmse - model_rmse)
print("% reduction in error:", (baseline_rmse - model_rmse) / baseline_rmse*100)

Baseline RMSE (prediciting mean): 1046895.5916412137
Model RMSE: 793342.8870673801
Improvement: 253552.70457383362
% reduction in error: 24.21948345167259


Compared to a naive baseline that predicts the mean price for every house, the trained linear regression model achieves RMSE OF 793K - a 24.2% reduction in error. This confirms the model adds predictive meaningful value beyond simply guessing the average. However, the remaining error and R² indicate an unexplained variance due to missing features(e.g. city).